# Tarea 9 — Anotación funcional y análisis de enriquecimiento
## Evaluación sumativa — 20 puntos

**Modalidad:** individual  
**Caso:** respuesta transcripcional a hipoxia  
**Datos reales:** NCBI GEO, **GSE280308**  
**Comparación:** células SCC25 en hipoxia vs. normoxia durante 12 h

### Objetivo
Aplicar de manera autónoma el flujo trabajado en el práctico demostrativo:

**datos → background → lista de interés → anotación → enriquecimiento GO-BP → dotplot → interpretación biológica**

> **Importante:** se evaluará tanto la ejecución correcta como la interpretación. Una figura o un bloque de código sin interpretación no asegura el puntaje completo.


## Contexto biológico

La disponibilidad reducida de oxígeno puede modificar la expresión génica celular. En este experimento se analizaron mediante RNA-seq células humanas **SCC25** mantenidas durante 12 horas en **normoxia** o **hipoxia**. El estudio contiene **3 réplicas biológicas por condición**.

### Pregunta de investigación de la tarea

> **¿Qué procesos biológicos están sobrerrepresentados entre los genes que aumentan su expresión en hipoxia respecto de normoxia?**

En esta evaluación trabajará con un experimento distinto al utilizado en el práctico de LPS.


# Preparación del entorno

Ejecute las siguientes celdas sin modificar su contenido.


In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")

paquetes <- c("DESeq2", "clusterProfiler", "org.Hs.eg.db", "enrichplot")
faltan <- paquetes[!sapply(paquetes, requireNamespace, quietly = TRUE)]
if (length(faltan) > 0) BiocManager::install(faltan, ask = FALSE, update = FALSE)

library(DESeq2)
library(clusterProfiler)
library(org.Hs.eg.db)
library(enrichplot)
library(ggplot2)


## Descargar los datos reales

El archivo de GEO contiene conteos y anotaciones del experimento. La celda siguiente descarga los datos y muestra sus primeras filas.


In [ ]:
url <- "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE280nnn/GSE280308/suppl/GSE280308_counts_anno.txt.gz"
archivo <- "GSE280308_counts_anno.txt.gz"

if (!file.exists(archivo)) download.file(url, archivo, mode = "wb")

datos <- read.delim(archivo, check.names = FALSE)
cat("Dimensiones del archivo:", dim(datos), "\n")
head(datos)
colnames(datos)


### Preparación del análisis diferencial

La siguiente celda identifica las columnas de normoxia e hipoxia, construye la matriz de conteos y realiza el análisis diferencial. **No se evalúa DESeq2 en esta tarea**; este bloque solamente genera la tabla que utilizará en las actividades evaluadas.

Si la detección automática de columnas no encuentra 3 muestras por condición, revise `colnames(datos)` antes de continuar.


In [ ]:
# Identificar las muestras
nombres <- colnames(datos)

# El archivo utiliza "hypoxia" y "nomoxia" en los nombres de las columnas
cols_hyp <- grep("hypoxia", nombres, ignore.case = TRUE, value = TRUE)
cols_norm <- grep("nomoxia", nombres, ignore.case = TRUE, value = TRUE)

cat("Columnas hipoxia:\n")
print(cols_hyp)

cat("\nColumnas normoxia:\n")
print(cols_norm)

# Verificar que existan 3 réplicas por condición
if (length(cols_hyp) != 3 || length(cols_norm) != 3) {
  stop("Revise los nombres de las columnas: se esperaban 3 muestras de hipoxia y 3 de normoxia.")
}

# Los genes están identificados mediante SYMBOL en la columna "id"
gene_id <- datos$id

# Construir matriz de conteos
counts <- as.matrix(
  datos[, c(cols_norm, cols_hyp)]
)

rownames(counts) <- gene_id

# Convertir a formato numérico
storage.mode(counts) <- "numeric"

# Verificar que los conteos sean no negativos
if (any(counts < 0, na.rm = TRUE)) {
  stop("Se detectaron valores negativos en la matriz de conteos.")
}

# DESeq2 requiere conteos enteros
if (any(abs(counts - round(counts)) > 1e-8, na.rm = TRUE)) {
  stop("La matriz contiene valores no enteros. No continúe con DESeq2.")
}

storage.mode(counts) <- "integer"

# Eliminar genes duplicados si existieran
counts <- counts[
  !duplicated(rownames(counts)),
  ,
  drop = FALSE
]

# Filtrar genes con muy pocos conteos
keep <- rowSums(counts) >= 10
counts_f <- counts[keep, , drop = FALSE]

cat("\nGenes antes del filtrado:", nrow(counts), "\n")
cat("Genes después del filtrado:", nrow(counts_f), "\n")


In [ ]:
# Definir las condiciones experimentales
condicion <- factor(
  c(rep("Normoxia", 3),
    rep("Hipoxia", 3)),
  levels = c("Normoxia", "Hipoxia")
)

coldata <- data.frame(
  condicion = condicion,
  row.names = colnames(counts_f)
)

# Crear objeto DESeq2
dds <- DESeqDataSetFromMatrix(
  countData = counts_f,
  colData = coldata,
  design = ~ condicion
)

# Análisis diferencial
dds <- DESeq(dds)

# Comparación: Hipoxia vs Normoxia
res <- results(
  dds,
  contrast = c("condicion", "Hipoxia", "Normoxia")
)

# Convertir resultados a data.frame
DE <- as.data.frame(res)

# Los nombres de fila corresponden a SYMBOL
DE$SYMBOL <- rownames(DE)

# Convertir SYMBOL a ENTREZID
DE$ENTREZID <- mapIds(
  org.Hs.eg.db,
  keys = DE$SYMBOL,
  column = "ENTREZID",
  keytype = "SYMBOL",
  multiVals = "first"
)

# Construir background
background_entrez <- unique(
  na.omit(DE$ENTREZID)
)

cat("Genes analizados después del filtrado:", nrow(DE), "\n")
cat("Genes del background con ENTREZID:", length(background_entrez), "\n")


# Actividad 1 — Construcción de la lista de interés (3 puntos)

Seleccione los genes **sobreexpresados en hipoxia** utilizando simultáneamente:

- `padj < 0.05`
- `log2FoldChange > 1`

**a)** Escriba el código utilizado e informe cuántos genes cumplen ambos criterios. **(1 punto)**  
**b)** Explique qué significa `log2FoldChange > 1` en la comparación Hipoxia vs. Normoxia. **(1 punto)**  
**c)** Explique por qué utilizamos `padj` en lugar de interpretar solamente el `pvalue`. **(1 punto)**


In [ ]:
# RESPUESTA
# Cree un objeto llamado DE_up y otro llamado genes_up.


### RESPUESTA — 1b y 1c





**1b)**

**1c)**


# Actividad 2 — Lista de interés y background (3 puntos)

**a)** Informe el número de genes de su **lista de interés** y el número de genes del **background** con ENTREZID. **(1 punto)**  
**b)** Explique con sus palabras la diferencia entre lista de interés y background. **(1 punto)**  
**c)** Explique por qué el background no debería corresponder simplemente a todos los genes humanos conocidos. **(1 punto)**


In [ ]:
# RESPUESTA
# Muestre aquí los tamaños solicitados.


### RESPUESTA — 2b y 2c

**2b)**

**2c)**


# Actividad 3 — Enriquecimiento GO-BP (5 puntos)

Realice un análisis de sobrerrepresentación utilizando:

- `enrichGO()`
- `gene = genes_up`
- `universe = background_entrez`
- organismo humano (`org.Hs.eg.db`)
- identificadores `ENTREZID`
- ontología `BP`
- método de ajuste `BH`
- `pvalueCutoff = 0.05`
- `readable = TRUE`

**a)** Escriba y ejecute correctamente el código. Guarde el resultado como `ego_up`. **(2 puntos)**  
**b)** Convierta el resultado a `data.frame` y muestre los cinco términos con menor `p.adjust`. **(1 punto)**  
**c)** Seleccione uno de esos términos e interprete correctamente `GeneRatio`, `BgRatio`, `Count` y `p.adjust`. **(2 puntos)**


In [ ]:
# RESPUESTA


### RESPUESTA-3c





**Término seleccionado:**

- **GeneRatio:**
- **BgRatio:**
- **Count:**
- **p.adjust:**
- **Interpretación conjunta:**


# Actividad 4 — Dotplot e interpretación (4 puntos)

Genere un dotplot mostrando los **12 términos principales**. **(1 punto)**

Luego responda:

**a)** ¿Qué término presenta el mayor `GeneRatio` entre los mostrados? **(1 punto)**  
**b)** ¿Qué representan el tamaño y el color de los puntos? **(1 punto)**  
**c)** ¿El término con mayor `GeneRatio` necesariamente debe presentar el menor `p.adjust`? Explique utilizando la lógica del enriquecimiento. **(1 punto)**


In [ ]:
# RESPUESTA


### RESPUESTAS — 4a, 4b y 4c

**4a)**

**4b)**

**4c)**


# Actividad 5 — Interpretación biológica (3 puntos)

A partir de **al menos dos términos enriquecidos** de sus resultados:

**a)** Proponga una interpretación de la respuesta biológica observada en las células expuestas a hipoxia. Sustente su respuesta con los resultados del análisis. **(2 puntos)**  
**b)** ¿Puede concluir únicamente a partir de este análisis que esos procesos están “activados”? Justifique. **(1 punto)**


### RESPUESTAS — Actividad 5

1.   Elemento de la lista
2.   Elemento de la lista



**5a)**

**5b)**


# Actividad 6 — Conclusión (2 puntos)

Redacte una conclusión de **máximo 120 palabras** que integre:

- la comparación realizada;
- los principales procesos enriquecidos;
- una interpretación biológica coherente;
- una formulación prudente sobre lo que el análisis permite concluir.

**Criterios:** síntesis e integración biológica **(1 punto)** + interpretación metodológicamente correcta **(1 punto)**.


### RESPUESTA — Conclusión

*(máximo 120 palabras)*


---

# Puntaje total: 20 puntos

| Actividad | Puntaje |
|---|---:|
| 1. Lista de interés | 3 |
| 2. Lista de interés y background | 3 |
| 3. Enriquecimiento GO-BP | 5 |
| 4. Dotplot e interpretación | 4 |
| 5. Interpretación biológica | 3 |
| 6. Conclusión | 2 |
| **Total** | **20** |

### Antes de entregar
Verifique que todas las celdas se ejecuten, que el dotplot sea visible, que sus respuestas estén completas y que la conclusión no exceda 120 palabras.
